[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/OMM-public/blob/master/cviceni/C4/notebooks/ct.ipynb)

# Rekonstrukce CT: 16 384 neznámých a jeden `backward()`

CT přístroj obraz neměří. Měří **projekce**: kolik útlumu nasbíral každý
rovnoběžný paprsek pod daným úhlem. Projekce pod všemi úhly dohromady tvoří
**sinogram** $\mathbf s$. Hledáme obraz $\mathbf x$ (128 × 128 pixelů), jehož
projekce sinogram zopakují:

$$
\text{minimize}_{\mathbf x}\ \ \frac{1}{m}\lVert P\mathbf x - \mathbf s\rVert^2
\;(+\,\lambda\,\mathrm{TV}(\mathbf x))
\qquad \text{subject to}\ \ 0 \le x_i \le 1
$$

Projekce $P$ = otočit obraz o úhel a sečíst po sloupcích. Gradient podle
všech pixelů najednou dá autograd, takže stačí napsat ztrátu a smyčku
z učení sítí. Doplňte **dvě místa `# TODO`**.

| ve vzorci | v kódu |
|---|---|
| $\mathbf x$ | `x` s `requires_grad=True` |
| $P\mathbf x$ | `projekce(x, u)` |
| $\mathbf s$ | `sino60`, `sino_sum` |
| $0 \le x_i \le 1$ | `x.clamp_(0, 1)` po každém kroku |

---

*Čísla, se kterými se notebook porovnává, počítají skripty [`kod/cisla.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C4/kod/cisla.py), [`kod/ct.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C4/kod/ct.py).*

In [ ]:
import os, urllib.request
RAW = "https://github.com/tomasvicar/OMM-public/raw/master/cviceni/C4/data"
if not os.path.exists("ct-128.png"):
    urllib.request.urlretrieve(f"{RAW}/ct-128.png", "ct-128.png")

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

ct = torch.from_numpy(np.asarray(Image.open("ct-128.png"), dtype=np.float32) / 255)[None, None]
N = ct.shape[-1]                               # 128

def uhly(pocet):
    return torch.arange(pocet) * math.pi / pocet          # rovnoměrně v <0, pi)

def projekce(obr, u):
    """Otočí obraz o každý úhel z u a sečte po sloupcích -> sinogram (úhly x N)."""
    c, s, z = torch.cos(u), torch.sin(u), torch.zeros_like(u)
    theta = torch.stack([torch.stack([c, -s, z], 1), torch.stack([s, c, z], 1)], 1)
    mrizka = F.affine_grid(theta, (len(u), 1, N, N), align_corners=False)
    otocene = F.grid_sample(obr.expand(len(u), 1, N, N), mrizka, align_corners=False)
    return otocene.sum(dim=2)[:, 0, :]

def tv(obr):
    """Celková variace: součet velikostí rozdílů sousedních pixelů (hladká verze)."""
    dx = obr[..., :, 1:] - obr[..., :, :-1]
    dy = obr[..., 1:, :] - obr[..., :-1, :]
    return torch.sqrt(dx[..., :-1, :] ** 2 + dy[..., :, :-1] ** 2 + 1e-6).sum()

yy, xx = torch.meshgrid(torch.linspace(-1, 1, N), torch.linspace(-1, 1, N), indexing="ij")
kruh = (xx ** 2 + yy ** 2 <= 1).float()
def rmse(x):                                   # chyba vůči originálu uvnitř kruhu
    return (((x - ct) ** 2 * kruh).sum() / kruh.sum()).sqrt().item()

u60 = uhly(60)
sino60 = projekce(ct, u60)                     # "naměřená" data, 60 x 128
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(ct[0, 0], cmap="gray"); ax[0].set_title("řez (ten hledáme)")
ax[1].imshow(sino60, cmap="gray", aspect="auto"); ax[1].set_title("sinogram (to známe)")
ax[1].set_xlabel("detektor"); ax[1].set_ylabel("úhel")
plt.show()

## Krok 1 — optimalizační smyčka *(díra 1)*

Funkce `rekonstruuj` dostane libovolnou ztrátu jako funkci obrazu. Smyčka je
stejná jako při učení sítě, jen „vahami“ jsou pixely. Numerický gradient by
na jeden krok potřeboval $2 \cdot 16\,384 = 32\,768$ projekcí, autograd jednu
projekci tam a jeden zpětný průchod.

In [ ]:
def rekonstruuj(ztrata, kroku=300, lr=0.05):
    """Najde obraz x (1 x 1 x N x N), který minimalizuje ztrata(x), pixely v <0, 1>."""
    x = torch.zeros(1, 1, N, N, requires_grad=True)     # start: černý obraz
    opt = torch.optim.Adam([x], lr=lr)
    for k in range(kroku):
        # TODO 1: jeden krok Adamu jako při učení sítě
        #   vynulovat gradient, spočítat ztrátu, backward, krok optimalizátoru
        ...
        # a pak pixely oříznout na <0, 1>  (pod torch.no_grad(), metoda clamp_)
        ...
    return x.detach()

In [ ]:
start = time.perf_counter()
x60 = rekonstruuj(lambda x: ((projekce(x, u60) - sino60) ** 2).mean())
print(f"60 úhlů: RMSE = {rmse(x60):.4f}  ({time.perf_counter() - start:.1f} s)")
assert abs(rmse(x60) - 0.0331) < 0.003, "nesedí s RMSE 0,0331 ze zadání"

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(x60[0, 0], cmap="gray", vmin=0, vmax=1); ax[0].set_title("rekonstrukce, 60 úhlů")
ax[1].imshow(ct[0, 0], cmap="gray", vmin=0, vmax=1); ax[1].set_title("originál")
plt.show()

## Krok 2 — málo úhlů, šum a penalizace TV *(díra 2)*

Nejdřív jen 12 úhlů (méně dávky pro pacienta). Pak zase 60 úhlů, ale se šumem
$\sigma = 1$ v sinogramu. Proti šumu pomůže přidat do ztráty **celkovou
variaci** $\mathrm{TV}(\mathbf x)$, která trestá skoky mezi sousedními pixely.
Gradient nového členu zase dodá autograd, nic se neodvozuje.

In [ ]:
u12 = uhly(12)
sino12 = projekce(ct, u12)
x12 = rekonstruuj(lambda x: ((projekce(x, u12) - sino12) ** 2).mean())
print(f"12 úhlů: RMSE = {rmse(x12):.4f}")

torch.manual_seed(0)
sino_sum = sino60 + 1.0 * torch.randn_like(sino60)          # šum sigma = 1
x_sum = rekonstruuj(lambda x: ((projekce(x, u60) - sino_sum) ** 2).mean())
print(f"šum, bez TV: RMSE = {rmse(x_sum):.4f}")

In [ ]:
LAM = 0.001

def ztrata_tv(x):
    # TODO 2: MSE vůči zašuměnému sinogramu + LAM krát celková variace tv(x)
    return ...

In [ ]:
x_tv = rekonstruuj(ztrata_tv)
print(f"šum, TV s lambda = {LAM}: RMSE = {rmse(x_tv):.4f}")
assert rmse(x_tv) < rmse(x_sum) - 0.01, "TV měla pomoct; je v ztrata_tv opravdu + LAM * tv(x)?"

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, obr, nazev in zip(ax, [x12, x_sum, x_tv], ["12 úhlů", "šum, bez TV", f"šum, TV, lambda = {LAM}"]):
    a.imshow(obr[0, 0], cmap="gray", vmin=0, vmax=1); a.set_title(f"{nazev}, RMSE {rmse(obr):.3f}")
plt.show()
# Zkuste LAM 10x větší a 10x menší: malá váha nechá šum, velká obraz rozmaže do plošek.